# HealthyAir - Feature Engineering Notebook
Run notebooks in order: exploration → cleaning → feature engineering, keeping the raw files unchanged.

## Feature engineering
The notebook creates time and lag features, measurements available at hour t to predict the NO2 measurement/category at hour t+1. 
A fixed seed makes resampling repeatable.

In [ ]:
# Import Libraries
import sys
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import statsmodels
import scipy
import os
import gcsfs
import warnings
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score,
    f1_score
)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from statsmodels.stats.proportion import proportion_confint

SEED = 42
rng = np.random.default_rng(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 110)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (7, 3.4)

def check(label, ok):
    """Self-check: prints PASS or CHECK AGAIN."""
    print(("PASS         " if bool(ok) else "CHECK AGAIN  ") + label)
    
# Confirm location and Libraries
#print(Path.cwd())
#print(PROJECT_ROOT)
print("Location:", sys.executable)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seaborn:", sns.__version__)
print("statsmodels:", statsmodels.__version__)

## Load Data & Validate

In [ ]:
# Healthy Air Project GCS Configuration
# ---------------------------------------------------------------------
GCS_PROJECT_ID = "healthy_air_analysis"
GCS_BUCKET = "uci_air_quality_20119102"
GCS_FILE = "gs://uci_air_quality_20119102/data/processed/.csv"

# Data paths
# ---------------------------------------------------------------------
RAW_GCS_PATH = "data_raw/uci-air-quality_20119102/v1/AirQualityUCI.csv"
PROCESSED_GCS_PATH = "processed/v1/"

PROJECT_ROOT = Path.cwd().parent
while not (PROJECT_ROOT / ".git").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURE_DIR = PROJECT_ROOT / "data" / "features"
FILE_PATH = PROCESSED_DIR/ "air_quality_cleaned.csv"
df = pd.read_csv(FILE_PATH)

# Data contract
required = {"Date", "Time", "CO(GT)", "PT08.S1(CO)", "NMHC(GT)", "C6H6(GT)",
            "PT08.S2(NMHC)", "NOx(GT)", "PT08.S3(NOx)", "NO2(GT)", "PT08.S4(NO2)", "PT08.S5(O3)", "T", "RH", "AH"}
assert required.issubset(df.columns)
#assert set(df["Quality"].unique()).issubset({0, 1, 3})
print("Initial data-contract checks passed.")
print(f"Project: {GCS_PROJECT_ID}")
print(f"Bucket: gs://{GCS_BUCKET}")
print("Dataset:", GCS_BUCKET)
print(f"Rows: {df.shape[0]:,}; columns: {df.shape[1]}")
display(df.head())

df_target = df.copy()

## 1. Define categories/ Air Quality Thresholds

Class Distribution and Imbalance Analysis

In [ ]:
# NO2 thresholds (µg/m³)
GOOD_THRESHOLD = 100
POOR_THRESHOLD = 200

# Ensure NO2 is numeric
df_target["NO2(GT)"] = pd.to_numeric(
    df["NO2(GT)"], errors="coerce"
)

# Convert UCI missing-value marker
#df_target["NO2(GT)"] = df_target["NO2(GT)"].replace(-200, np.nan)

# Three air-quality categories
conditions = [
    df_target["NO2(GT)"] < GOOD_THRESHOLD,
    (df_target["NO2(GT)"] >= GOOD_THRESHOLD) &
    (df_target["NO2(GT)"] <= POOR_THRESHOLD),
    df_target["NO2(GT)"] > POOR_THRESHOLD
]

labels = ["GOOD", "MODERATE", "POOR"]

df_target["NO2_Category"] = np.select(
    conditions,
    labels,
    default=None
)

# Check results
print(df_target[["NO2(GT)", "NO2_Category"]].head(10))

print("\nCategory counts:")
print(df_target["NO2_Category"].value_counts(dropna=False))

print("\nCategory percentages:")
print(
    df_target["NO2_Category"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)


In [ ]:
# Visualize
# -----------------------------------------------------------------------
plt.figure(figsize=(9, 5))

sns.countplot(
    data=df_target,
    x="NO2_Category",
    order=["GOOD", "MODERATE", "POOR"]
)

plt.title("Distribution of NO2 Air Quality Categories")
plt.xlabel("NO2 Category")
plt.ylabel("Number of observations")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 2. Define the next-hour target
A simple `.shift(-1)` is only a true next-hour target if the next row is exactly one hour later. Mask gaps to avoid accidentally predicting several hours ahead.

## OR

In [ ]:
df_target["Target_Next_Hour"] = df_target["NO2_Category"].shift(-1)

## 2. Lag and rolling features
These use past measurements only. The shift-before-rolling operation prevents the current row from entering its own rolling mean. Validate time gaps before relying on row-based lags.

In [ ]:
# Select numerical measurement variables
measurement_cols = [
    "CO(GT)", "PT08.S1(CO)", "NMHC(GT)", "C6H6(GT)",
    "PT08.S2(NMHC)", "NOx(GT)", "PT08.S3(NOx)",
    "NO2(GT)", "PT08.S4(NO2)", "PT08.S5(O3)",
    "T", "RH", "AH"
]

In [ ]:
lag_cols = [c for c in measurement_cols if c in df.columns]
for col in lag_cols:
    df[f"{col}_lag_1"] = df[col].shift(1)
    df[f"{col}_lag_2"] = df[col].shift(2)
    df[f"{col}_rolling_mean_3"] = df[col].shift(1).rolling(3, min_periods=3).mean()
display(df[["Datetime"] + [f"{c}_lag_1" for c in lag_cols]].head())

In [ ]:
# Define project-specific NO2 categories
forecast_df["Target_Next_Hour"] = pd.cut(
    df_target["NO2_next_hour"],
    bins=[-np.inf, 100, 200, np.inf],
    labels=["GOOD", "MODERATE", "POOR"]
)

# Candidate predictors: current measurements only
predictors = measurement_cols.copy()

# Prepare modelling dataset
model_df = forecast_df[
    ["Datetime"] + predictors + ["Target_Next_Hour"]
].dropna(subset=["Target_Next_Hour"])

# Chronological 80/20 train-test split
split_index = int(len(model_df) * 0.8)

train = model_df.iloc[:split_index]
test = model_df.iloc[split_index:]

X_train = train[predictors]
X_test = test[predictors]

y_train = train["Target_Next_Hour"]
y_test = test["Target_Next_Hour"]

# Remove predictors completely missing in the training data
usable_predictors = X_train.columns[
    X_train.notna().any()
].tolist()

X_train = X_train[usable_predictors]
X_test = X_test[usable_predictors]

# Baseline: always predict the most frequent training category
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

# Random Forest pipeline
rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)

rf_predictions = rf_pipeline.predict(X_test)

# Compare model performance
results = pd.DataFrame({
    "model": ["Baseline", "Random Forest"],
    "accuracy": [
        baseline.score(X_test, y_test),
        rf_pipeline.score(X_test, y_test)
    ],
    "balanced_accuracy": [
        balanced_accuracy_score(y_test, baseline_predictions),
        balanced_accuracy_score(y_test, rf_predictions)
    ],
    "macro_f1": [
        f1_score(y_test, baseline_predictions, average="macro"),
        f1_score(y_test, rf_predictions, average="macro")
    ]
})

display(results.round(3))

# Detailed Random Forest results
print("Random Forest classification report:")
print(classification_report(
    y_test,
    rf_predictions,
    zero_division=0
))
